In [2]:
!pip install -q ultralytics torch torchvision pillow opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.2 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from ultralytics import YOLO
import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [6]:
yolo_model = YOLO("/content/best.pt")


In [ ]:
NUM_CLASSES = 25

classifier = models.efficientnet_b0(pretrained=False)
classifier.classifier[1] = nn.Linear(
    classifier.classifier[1].in_features,
    NUM_CLASSES
)

classifier.load_state_dict(
    torch.load("efficientnetb0_final.pth", map_location=device)
)

classifier = classifier.to(device)
classifier.eval()


In [7]:
CLASS_NAMES =['airplane',
 'bed',
 'bench',
 'bicycle',
 'bird',
 'bottle',
 'bowl',
 'bus',
 'cake',
 'car',
 'cat',
 'chair',
 'couch',
 'cow',
 'cup',
 'dog',
 'elephant',
 'horse',
 'motorcycle',
 'person',
 'pizza',
 'potted plant',
 'stop sign',
 'traffic light',
 'truck']

In [8]:
clf_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [19]:
image_path = "/content/airplane_272.jpg"

results = yolo_model.predict(
    source=image_path,
    conf=0.1,     # lower threshold
    imgsz=640,
    device=0
)



image 1/1 /content/airplane_272.jpg: 640x448 1 motorcycle, 97.9ms
Speed: 4.8ms preprocess, 97.9ms inference, 42.1ms postprocess per image at shape (1, 3, 640, 448)


In [20]:
!ls


1.jpg  airplane_272.jpg  best.pt  efficientnetb0_final.pth  sample_data


In [23]:
import torch
import torch.nn as nn
from torchvision import models

NUM_CLASSES = 25

classifier = models.efficientnet_b0(pretrained=False)
classifier.classifier[1] = nn.Linear(
    classifier.classifier[1].in_features,
    NUM_CLASSES
)

classifier.load_state_dict(
    torch.load("efficientnetb0_final.pth", map_location=device)
)

classifier = classifier.to(device)
classifier.eval()


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [25]:

print("Number of detections:", len(results[0].boxes))
image_path = "/content/airplane_272.jpg"
img = Image.open(image_path).convert("RGB")
img_rgb = np.array(img)

if len(results[0].boxes) == 0:
    print("⚠️ YOLO detected NO objects in this image.")
else:
    print("✅ YOLO detected objects, running classification...")

    for box in results[0].boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0])

        crop = img_rgb[y1:y2, x1:x2]
        if crop.size == 0:
            continue

        pil_crop = Image.fromarray(crop)
        input_tensor = clf_transform(pil_crop).unsqueeze(0).to(device)

        with torch.no_grad():
            output = classifier(input_tensor)
            pred_id = torch.argmax(output, dim=1).item()
            pred_label = CLASS_NAMES[pred_id]

        print("Predicted:", pred_label)

        cv2.rectangle(img_rgb, (x1,y1), (x2,y2), (0,255,0), 2)
        cv2.putText(
            img_rgb,
            pred_label,
            (x1, y1-10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255,0,0),
            2
        )


Number of detections: 1
✅ YOLO detected objects, running classification...
Predicted: motorcycle
